# Measure the political shift with B

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch


tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
model = AutoModelForSequenceClassification.from_pretrained("bucketresearch/politicalBiasBERT")
model.eval()


def compute_political_stance(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=-1)[0]  # [P(left), P(center), P(right)]
    anchors = torch.tensor([-1.0, 0.0, 1.0])
    score = (probs * anchors).sum().item()
    return score


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [9]:
import random
import re
from pathlib import Path

# --- Paths ---------------------------------------------------------------
REPO_ROOT = Path("/Users/kunalnarwani/Desktop/Delft/Delft/Sem_1/NLP for society/Group_project/code/moral-summarization")
SUMMARIES_DIR = REPO_ROOT / "results" / "summaries" / "allsides"

# --- Regexes to strip the prompt/response scaffolding -------------------
ARTICLE_RE = re.compile(
    r"Here is the news article:\s*(.*?)\s*The summary has to be returned",
    re.DOTALL,
)
SUMMARY_RE = re.compile(r"SUMMARY:\s*(.*?)\s*END OF SUMMARY\.", re.DOTALL)


def load_random_article_and_summary(
    rng: random.Random | None = None,
    method: str | None = None,
    model: str | None = None,
):
    """Pick one random AllSides article + one random model summary for it.

    If method/model are None, they're sampled uniformly from what's on disk.
    Returns a dict with article_id, leaning, article_text, summary_text,
    plus the metadata (method, model, seed) of the chosen summary.
    """
    rng = rng or random.Random()

    # 1. Random article folder
    article_dirs = [p for p in SUMMARIES_DIR.iterdir() if p.is_dir()]
    article_dir = rng.choice(article_dirs)
    article_id = article_dir.name
    leaning = {"l": "left", "c": "center", "r": "right"}[article_id.split("_")[-2]]

    # 2. Extract the source article from any prompt file (they all embed the same text;
    #    vanilla_prompt.txt is the simplest scaffold).
    prompt_text = (article_dir / "vanilla_prompt.txt").read_text()
    m = ARTICLE_RE.search(prompt_text)
    if not m:
        raise ValueError(f"Couldn't find article body in {article_dir}/vanilla_prompt.txt")
    article_text = m.group(1).strip()

    # 3. Pick a random response file (any method, any model, any seed)
    response_files = list(article_dir.glob("*_response_*.txt"))
    if method is not None:
        response_files = [p for p in response_files if p.name.startswith(f"{method}_response_")]
    if model is not None:
        response_files = [p for p in response_files if f"_response_{model}_" in p.name]
    if not response_files:
        raise ValueError(f"No response files match method={method}, model={model} in {article_dir}")
    response_path = rng.choice(response_files)

    # 4. Parse method, model, seed back out of the filename
    fname = response_path.stem  # e.g. "vanilla_response_Meta-Llama-3-70B-Instruct_345"
    method_name, _, rest = fname.partition("_response_")
    model_name, _, seed = rest.rpartition("_")

    # 5. Extract the summary body
    response_text = response_path.read_text()
    m = SUMMARY_RE.search(response_text)
    summary_text = m.group(1).strip() if m else response_text.strip()  # fallback: raw text

    return {
        "article_id": article_id,
        "leaning": leaning,
        "method": method_name,
        "model": model_name,
        "seed": seed,
        "article_text": article_text,
        "summary_text": summary_text,
    }


# --- Example use --------------------------------------------------------
rng = random.Random(42)  # set a seed for reproducibility, or omit for true random
sample = load_random_article_and_summary(rng=rng)

print(f"Article ID : {sample['article_id']}  ({sample['leaning']})")
print(f"Method     : {sample['method']}")
print(f"Model      : {sample['model']}  (seed {sample['seed']})")
print(f"\n--- ARTICLE ({len(sample['article_text'])} chars) ---")
print(sample['article_text'][:500] + " ...")


article_score = compute_political_stance(sample['article_text'])
print("Political stance score of the article body:")
print(f"{article_score:+.3f}")

print(f"\n--- SUMMARY ({len(sample['summary_text'])} chars) ---")
print(sample['summary_text'])
summary_score = compute_political_stance(sample['summary_text'])
print("Political stance score of the summary:")
print(f"{summary_score:+.3f}")

Article ID : allsides_abortion_c_3  (center)
Method     : cot
Model      : Meta-Llama-3-70B-Instruct  (seed 345)

--- ARTICLE (3192 chars) ---
The Alabama Senate on Tuesday evening passed legislation that bans nearly all abortions in the state, sending the measure to Gov. Kay Ivey's (R) desk to be signed into law. Ivey has not said whether she will sign the measure, which passed by a 25-6 margin. It would ban abortions in virtually all instances in Alabama, including for victims of rape and incest, and would only permit the procedure if necessary to save a mother’s life. Anyone performing an abortion could be punished by 10 to 99 years ...
Political stance score of the article body:
+0.918

--- SUMMARY (739 chars) ---
The Alabama Senate has passed a highly restrictive abortion law, banning nearly all abortions in the state, including for victims of rape and incest. The law, which would punish doctors with up to 99 years in prison, is considered the most restrictive in the country. Prop